# 02 — Phase-1 Baseline Training (YOLOPv2-aligned)

**Two baselines live here.** Flip `BASELINE` in cell 4 to pick one.
Both share dataset, loss stack, and training loop; model + YAML +
checkpoint directory change per baseline.

| BASELINE | model source | aug | optimizer | LR | epochs | YAML |
|---|---|---|---|---|---|---|
| `'YOLOP'`   | YOLOP MCnet_0 minus DA head | letterbox + perspective + HSV + flip | Adam | 1e-3 | 100 | `configs/yolop_vehicle_lane_baseline.yaml` |
| `'YOLOPv2'` | YOLOP MCnet_0 + ELAN/groups + deconv lane head | + Mosaic + MixUp | **SGD** | **1e-2** | **300** | `configs/yolopv2_vehicle_lane_baseline.yaml` |

The YOLOPv2 protocol tracks the paper (arXiv 2208.11434 §3). Every
delta versus YOLOP is documented in `AUDIT_YOLOPV2_ALIGNMENT.md`.
Components we had to infer because YOLOPv2's public release is
inference-only are labelled `[INFERRED]` in the code.

**Output** (per baseline):
- `checkpoints/{baseline}/latest.pth`, `best.pth` on Drive
- TensorBoard logs under `tb_logs/{baseline}/`
- Metrics under `metrics/{baseline}/`

**Note on validation letterbox:** the YOLOPv2 paper uses 640×640 train
and 640×384 test. The validation loader below picks `cfg.TEST.IMAGE_SIZE`
when present; the YOLOPv2 YAML sets `[640, 384]`. YOLOP baseline YAML
keeps 640 for both.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yacs tqdm opencv-python-headless tensorboard

In [ ]:
import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import logging
import math
from pathlib import Path
from torch import amp  # torch.amp replaces torch.cuda.amp in recent PyTorch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, train, validate
from lib.dataset import BddDataset

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Baseline selection + config ──
# BASELINE: 'YOLOP' or 'YOLOPv2'. Switches model, yaml, and checkpoint dir.
BASELINE = 'YOLOPv2'

from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

yaml_map = {
    'YOLOP':   os.path.join(REPO_ROOT, 'configs', 'yolop_vehicle_lane_baseline.yaml'),
    'YOLOPv2': os.path.join(REPO_ROOT, 'configs', 'yolopv2_vehicle_lane_baseline.yaml'),
}
cfg.defrost()
cfg.merge_from_file(yaml_map[BASELINE])

# Resolve dataset roots (Colab SSD -> Drive fallback).
ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
# Raw BDD may be `raw/images/100k/...` OR the flat `raw/100k/...` —
# go through the resolver, never hardcode.
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)

cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')

# Drive persistence — per-baseline subdirs so the two runs don't collide.
cfg.DRIVE.ROOT = ECOCAR_ROOT
cfg.DRIVE.CHECKPOINT_DIR = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'checkpoints', BASELINE.lower())
cfg.DRIVE.METRICS_DIR = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'metrics', BASELINE.lower())

# Pin the model name in case the yaml omits it.
cfg.MODEL.NAME = BASELINE
cfg.freeze()

os.makedirs(cfg.DRIVE.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.DRIVE.METRICS_DIR, exist_ok=True)
output_dir = cfg.DRIVE.METRICS_DIR
tb_log_dir = os.path.join(cfg.DRIVE.ROOT, 'yolop_vehicle_lane', 'tb_logs', BASELINE.lower())
os.makedirs(tb_log_dir, exist_ok=True)

print(f'Baseline     : {BASELINE}')
print(f'BDD_IMAGES   : {BDD_IMAGES}')
print(f'BDD_LABELS   : {BDD_LABELS}')
print(f'Mosaic       : {cfg.DATASET.MOSAIC}')
print(f'MixUp        : {cfg.DATASET.MIXUP}')
print(f'Checkpoints  : {cfg.DRIVE.CHECKPOINT_DIR}')

In [ ]:
# ── Logger ──
logger = logging.getLogger('train')
logger.setLevel(logging.INFO)
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)

In [ ]:
# ── Build datasets ──
# YOLOPv2 paper §3: train 640×640 letterbox, test 640×384 letterbox.
# MODEL.IMAGE_SIZE is written (W, H) in the YAML. BddDataset now accepts
# either an int (square auto-letterbox) or an explicit (H, W) tuple
# (rectangular letterbox with auto=False).
transform = T.ToTensor()

train_wh = tuple(cfg.MODEL.IMAGE_SIZE)              # (W, H)
val_wh = tuple(getattr(cfg.TEST, 'IMAGE_SIZE', cfg.MODEL.IMAGE_SIZE))

train_size = int(max(train_wh))                     # 640 — square aug-friendly
val_size = (int(val_wh[1]), int(val_wh[0]))         # (H, W) for letterbox auto=False

train_dataset = BddDataset(cfg, is_train=True,  inputsize=train_size, transform=transform)
val_dataset   = BddDataset(cfg, is_train=False, inputsize=val_size,   transform=transform)

train_loader = DataLoader(
    train_dataset, batch_size=cfg.TRAIN.BATCH_SIZE_PER_GPU,
    shuffle=cfg.TRAIN.SHUFFLE, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=train_dataset.collate_fn,
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
    shuffle=False, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=val_dataset.collate_fn,
)

# Sanity check — one sample tells us the actual letterboxed HxW used at train vs eval.
sample_train = train_dataset[0][0].shape
sample_val = val_dataset[0][0].shape
print(f'Train: {len(train_dataset)} samples, letterbox C×H×W = {tuple(sample_train)}')
print(f'Val:   {len(val_dataset)} samples, letterbox C×H×W = {tuple(sample_val)}')

In [ ]:
# ── Build model, loss, optimizer, scheduler ──
# Scheduler policy
# ----------------
# YOLOPv2 paper §3 describes "cosine annealing with warm restart". Two
# interpretations are compatible with that phrasing:
#   (A) Per-iteration **linear warmup** for the first `WARMUP_EPOCHS`
#       epochs (the "warm restart" = warming up from zero), followed by
#       per-epoch **cosine annealing** from LR0 to LR0 * LRF over
#       the remaining epochs.
#   (B) True **SGDR** (CosineAnnealingWarmRestarts) with periodic
#       restarts throughout training.
#
# The paper does not publish `T_0` / `T_mult` or any restart period, so
# the SGDR variant would require inventing those. We default to (A) —
# which is YOLOP's own scheduler, matches the plain-English paper
# phrasing, and is well-calibrated to the 300-epoch budget — and expose
# `TRAIN.SGDR` + `TRAIN.SGDR_T0` knobs so a future ablation can swap in
# (B) without touching model / loss code. Marked [INFERRED] either way.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = get_net(cfg).to(device)
model.gr = 1.0
model.nc = 5
model.names = cfg.MODEL.VEHICLE_CLASSES

criterion = get_loss(cfg, device)

if cfg.TRAIN.OPTIMIZER == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.TRAIN.LR0,
                                 betas=(cfg.TRAIN.MOMENTUM, 0.999))
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.TRAIN.LR0,
                                momentum=cfg.TRAIN.MOMENTUM,
                                weight_decay=cfg.TRAIN.WD,
                                nesterov=cfg.TRAIN.NESTEROV)
for pg in optimizer.param_groups:
    pg['initial_lr'] = cfg.TRAIN.LR0

use_sgdr = bool(getattr(cfg.TRAIN, 'SGDR', False))
if use_sgdr:
    sgdr_t0 = int(getattr(cfg.TRAIN, 'SGDR_T0', max(1, cfg.TRAIN.END_EPOCH // 3)))
    sgdr_tmult = int(getattr(cfg.TRAIN, 'SGDR_TMULT', 1))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=sgdr_t0, T_mult=sgdr_tmult,
        eta_min=cfg.TRAIN.LR0 * cfg.TRAIN.LRF,
    )
    sched_desc = f'SGDR (T_0={sgdr_t0}, T_mult={sgdr_tmult})'
else:
    lf = lambda x: ((1 + math.cos(x * math.pi / cfg.TRAIN.END_EPOCH)) / 2) * \
                   (1 - cfg.TRAIN.LRF) + cfg.TRAIN.LRF
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)
    sched_desc = 'cosine-annealing + linear warmup (YOLOP default)'

scaler = amp.GradScaler(device.type, enabled=device.type != 'cpu')

num_params = sum(p.numel() for p in model.parameters())
print(f'Model params : {num_params/1e6:.2f}M')
print(f'Device       : {device}')
print(f'Optimizer    : {cfg.TRAIN.OPTIMIZER} @ LR0={cfg.TRAIN.LR0}, WD={cfg.TRAIN.WD}')
print(f'Scheduler    : {sched_desc}')
print(f'Warmup       : {cfg.TRAIN.WARMUP_EPOCHS} epochs (iter-based linear in-loop)')

In [ ]:
# ── Resume from checkpoint if available ──
start_epoch = cfg.TRAIN.BEGIN_EPOCH
best_map = 0.0
best_ll_iou = 0.0

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(ckpt_path) and cfg.AUTO_RESUME:
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_map = ckpt.get('best_map', 0.0)
    best_ll_iou = ckpt.get('best_ll_iou', 0.0)
    print(f'Resumed from epoch {start_epoch}, best_map={best_map:.4f}, best_ll_iou={best_ll_iou:.4f}')
else:
    print('Training from scratch')

In [ ]:
# ── Training loop ──
writer = SummaryWriter(tb_log_dir)
writer_dict = {
    'writer': writer,
    'train_global_steps': start_epoch * len(train_loader),
}

num_batch = len(train_loader)
num_warmup = max(round(cfg.TRAIN.WARMUP_EPOCHS * num_batch), 1000)

for epoch in range(start_epoch, cfg.TRAIN.END_EPOCH):
    # Train
    train(cfg, train_loader, model, criterion, optimizer, scaler,
          epoch, num_batch, num_warmup, writer_dict, logger, device)

    scheduler.step()

    # Validate
    if epoch % cfg.TRAIN.VAL_FREQ == 0:
        ll_seg_result, det_result, val_loss, maps, times = validate(
            epoch, cfg, val_loader, val_dataset, model, criterion,
            output_dir, tb_log_dir, writer_dict, logger, device
        )

        ll_acc, ll_iou, ll_miou = ll_seg_result
        mp, mr, map50, map_all = det_result

        # Log to tensorboard
        writer.add_scalar('val/loss', val_loss, epoch)
        writer.add_scalar('val/mAP50', map50, epoch)
        writer.add_scalar('val/mAP', map_all, epoch)
        writer.add_scalar('val/ll_iou', ll_iou, epoch)
        writer.add_scalar('val/ll_acc', ll_acc, epoch)

        logger.info(f'Epoch {epoch} | mAP50={map50:.4f} mAP={map_all:.4f} | '
                    f'LL_IoU={ll_iou:.4f} LL_Acc={ll_acc:.4f} | Loss={val_loss:.4f}')

        # Save checkpoint
        is_best = map50 > best_map or ll_iou > best_ll_iou
        if map50 > best_map:
            best_map = map50
        if ll_iou > best_ll_iou:
            best_ll_iou = ll_iou

        state = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_map': best_map,
            'best_ll_iou': best_ll_iou,
        }
        torch.save(state, os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth'))
        if is_best:
            torch.save(state, os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth'))
            logger.info(f'  *** New best: mAP50={best_map:.4f}, LL_IoU={best_ll_iou:.4f}')

writer.close()
print(f'\nTraining complete. Best mAP50={best_map:.4f}, Best LL_IoU={best_ll_iou:.4f}')